# HiPO IPM vs PuLP/CBC

This notebook compares the project's interior-point solver with the **HiPO LDLT** backend against PuLP's CBC solver. It checks objective agreement and primal feasibility on deterministic bounded LPs.

The suite covers equality constraints, mixed inequalities, shifted finite bounds, objective agreement, and independently measured primal feasibility.

## Prerequisites

Build the Python extension and install PuLP in the notebook environment if needed:

```bash
cmake --build build --target ipm_ext -j
python -m pip install pulp
```

In [1]:
from pathlib import Path
import importlib.metadata
import sys
import time

import numpy as np


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "CMakeLists.txt").is_file() and (candidate / "ipm").is_dir():
            return candidate
    raise RuntimeError("Could not find the solvers repository root from the current directory")


ROOT = find_repo_root(Path.cwd())
BUILD = ROOT / "build"
sys.path.insert(0, str(BUILD))

try:
    import ipm_solver
except ImportError as exc:
    raise RuntimeError(
        f"Could not import ipm_solver from {BUILD}. "
        "Build it with: cmake --build build --target ipm_ext -j"
    ) from exc

try:
    import pulp
except ImportError as exc:
    raise RuntimeError(
        "PuLP is required for this comparison. Install it with: python -m pip install pulp"
    ) from exc

cbc = pulp.PULP_CBC_CMD(msg=False)
if not cbc.available():
    raise RuntimeError("PuLP is installed, but its CBC executable is unavailable")

print(f"repository : {ROOT}")
print(f"extension  : {Path(ipm_solver.__file__).resolve()}")
print(f"PuLP       : {importlib.metadata.version('pulp')}")
print(f"CBC        : {cbc.path}")

repository : /data/dev/solvers
extension  : /data/dev/solvers/build/ipm_solver.cpython-314-x86_64-linux-gnu.so
PuLP       : 3.3.2
CBC        : /data/dev/solvers/.venv/lib/python3.14/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc


## Problem generation

Each random problem is feasible by construction: a point strictly inside finite bounds is sampled first, then the right-hand side is derived from it. Finite bounds also make every problem bounded.

In [2]:
def make_equality_lp(seed: int, n: int = 12, m: int = 5) -> dict:
    rng = np.random.default_rng(seed)
    lb = rng.uniform(-1.0, 0.0, size=n)
    ub = lb + rng.uniform(2.0, 6.0, size=n)
    x_feasible = lb + rng.uniform(0.2, 0.8, size=n) * (ub - lb)
    A = rng.normal(size=(m, n))
    return {
        "name": f"equality_seed_{seed}",
        "A": np.ascontiguousarray(A, dtype=np.float64),
        "b": np.asarray(A @ x_feasible, dtype=np.float64),
        "c": np.asarray(rng.normal(size=n), dtype=np.float64),
        "lb": np.asarray(lb, dtype=np.float64),
        "ub": np.asarray(ub, dtype=np.float64),
        "sense": ["="] * m,
    }


def make_mixed_lp(seed: int, n: int = 12, m: int = 6) -> dict:
    problem = make_equality_lp(seed, n=n, m=m)
    rng = np.random.default_rng(seed + 10_000)
    senses = np.array(["=", "<=", ">=", "<=", ">=", "="], dtype=object)
    x_feasible = problem["lb"] + 0.5 * (problem["ub"] - problem["lb"])
    ax = problem["A"] @ x_feasible
    slack = rng.uniform(0.2, 1.0, size=m)
    problem["b"] = np.where(senses == "<=", ax + slack, np.where(senses == ">=", ax - slack, ax))
    problem["sense"] = senses.tolist()
    problem["name"] = f"mixed_seed_{seed}"
    return problem


equality_problems = [make_equality_lp(seed) for seed in range(5)]
mixed_problems = [make_mixed_lp(seed) for seed in range(3)]
print(f"generated {len(equality_problems)} strict cases and {len(mixed_problems)} diagnostic cases")

generated 5 strict cases and 3 diagnostic cases


## Solver adapters and independent residual checks

Both adapters solve the same minimization model. The HiPO adapter explicitly selects `HIPO_LDLT`; it never relies on the IPM's default backend.

In [3]:
def solve_hipo(problem: dict, tol: float = 1e-8) -> dict:
    solver = ipm_solver.ip_solver()
    solver.set_solver_type(ipm_solver.SolverType.HIPO_LDLT)
    started = time.perf_counter()
    solution = solver.solve(
        problem["A"], problem["b"], problem["c"],
        problem["lb"], problem["ub"], problem["sense"], tol,
    )
    return {
        "solver": "IPM/HiPO",
        "status": solution.status,
        "objective": float(solution.objective),
        "x": np.asarray(solution.x, dtype=np.float64),
        "time_ms": 1e3 * (time.perf_counter() - started),
    }


def solve_pulp(problem: dict) -> dict:
    model = pulp.LpProblem(problem["name"], pulp.LpMinimize)
    xs = [
        pulp.LpVariable(f"x_{j}", lowBound=float(lo), upBound=float(hi))
        for j, (lo, hi) in enumerate(zip(problem["lb"], problem["ub"]))
    ]
    model += pulp.lpSum(float(coef) * x for coef, x in zip(problem["c"], xs))
    for i, (row, rhs, sense) in enumerate(zip(problem["A"], problem["b"], problem["sense"])):
        expression = pulp.lpSum(float(coef) * x for coef, x in zip(row, xs))
        if sense == "=":
            model += expression == float(rhs), f"row_{i}"
        elif sense == "<=":
            model += expression <= float(rhs), f"row_{i}"
        elif sense == ">=":
            model += expression >= float(rhs), f"row_{i}"
        else:
            raise ValueError(f"unsupported constraint sense: {sense!r}")

    started = time.perf_counter()
    status_code = model.solve(cbc)
    elapsed_ms = 1e3 * (time.perf_counter() - started)
    status = pulp.LpStatus[status_code]
    if status != "Optimal":
        return {"solver": "PuLP/CBC", "status": status, "objective": np.nan, "x": None, "time_ms": elapsed_ms}
    x = np.asarray([pulp.value(variable) for variable in xs], dtype=np.float64)
    return {
        "solver": "PuLP/CBC",
        "status": status,
        "objective": float(problem["c"] @ x),
        "x": x,
        "time_ms": elapsed_ms,
    }


def max_primal_violation(problem: dict, x: np.ndarray) -> float:
    if x is None or not np.all(np.isfinite(x)):
        return np.inf
    lhs = problem["A"] @ x
    violations = [np.max(np.maximum(problem["lb"] - x, 0.0)), np.max(np.maximum(x - problem["ub"], 0.0))]
    for value, rhs, sense in zip(lhs, problem["b"], problem["sense"]):
        if sense == "=":
            violations.append(abs(value - rhs))
        elif sense == "<=":
            violations.append(max(value - rhs, 0.0))
        else:
            violations.append(max(rhs - value, 0.0))
    return float(max(violations))


def compare(problem: dict) -> dict:
    hipo = solve_hipo(problem)
    reference = solve_pulp(problem)
    scale = 1.0 + abs(reference["objective"])
    return {
        "problem": problem["name"],
        "hipo_status": hipo["status"],
        "pulp_status": reference["status"],
        "hipo_obj": hipo["objective"],
        "pulp_obj": reference["objective"],
        "relative_obj_gap": abs(hipo["objective"] - reference["objective"]) / scale,
        "hipo_violation": max_primal_violation(problem, hipo["x"]),
        "pulp_violation": max_primal_violation(problem, reference["x"]),
        "hipo_ms": hipo["time_ms"],
        "pulp_ms": reference["time_ms"],
        "x_inf_distance": float(np.linalg.norm(hipo["x"] - reference["x"], ord=np.inf)),
    }


def print_results(rows: list[dict]) -> None:
    header = f"{'problem':<18} {'HiPO obj':>13} {'PuLP obj':>13} {'rel gap':>11} {'HiPO viol':>11} {'HiPO ms':>9} {'PuLP ms':>9}"
    print(header)
    print("-" * len(header))
    for row in rows:
        print(
            f"{row['problem']:<18} {row['hipo_obj']:>13.6g} {row['pulp_obj']:>13.6g} "
            f"{row['relative_obj_gap']:>11.3e} {row['hipo_violation']:>11.3e} "
            f"{row['hipo_ms']:>9.3f} {row['pulp_ms']:>9.3f}"
        )

## Strict equality-form regression suite

Timing is reported only as a smoke measurement: these models are too small for a meaningful performance comparison, and CBC process startup dominates its numbers.

In [4]:
strict_results = [compare(problem) for problem in equality_problems]
print_results(strict_results)

OBJECTIVE_RTOL = 1e-6
FEASIBILITY_ATOL = 1e-6

for result in strict_results:
    assert result["pulp_status"] == "Optimal", result
    assert result["hipo_status"] == "ipm", result
    assert np.isfinite(result["hipo_obj"]), result
    assert result["relative_obj_gap"] <= OBJECTIVE_RTOL, result
    assert result["hipo_violation"] <= FEASIBILITY_ATOL, result

print(f"\nPASS: {len(strict_results)} HiPO solutions agree with PuLP/CBC")

problem                 HiPO obj      PuLP obj     rel gap   HiPO viol   HiPO ms   PuLP ms
------------------------------------------------------------------------------------------
equality_seed_0         -15.7381      -15.7381   1.362e-09   2.461e-10   117.801     3.118
equality_seed_1         -19.2153      -19.2153   7.104e-10   5.321e-10     3.876     2.902
equality_seed_2          -17.812       -17.812   5.717e-09   2.087e-09     4.249     2.680
equality_seed_3         -8.75682      -8.75682   1.130e-08   2.777e-09     1.946     2.318
equality_seed_4           4.9104        4.9104   1.809e-09   2.227e-09     3.283     2.325

PASS: 5 HiPO solutions agree with PuLP/CBC


[IPM] Solver: HiPOLDLT (multifrontal BK+iterative refinement)
[IPM] Solver: HiPOLDLT (multifrontal BK+iterative refinement)
[IPM] Solver: HiPOLDLT (multifrontal BK+iterative refinement)
[IPM] Solver: HiPOLDLT (multifrontal BK+iterative refinement)
[IPM] Solver: HiPOLDLT (multifrontal BK+iterative refinement)


## Mixed-inequality regression suite

These cases exercise `<=` and `>=` standard-form conversion in addition to the HiPO solve.

In [5]:
mixed_results = [compare(problem) for problem in mixed_problems]
print_results(mixed_results)

for result in mixed_results:
    assert result["pulp_status"] == "Optimal", result
    assert result["hipo_status"] == "ipm", result
    assert np.isfinite(result["hipo_obj"]), result
    assert result["relative_obj_gap"] <= OBJECTIVE_RTOL, result
    assert result["hipo_violation"] <= FEASIBILITY_ATOL, result

print(f"\nPASS: all {len(mixed_results)} mixed-sense cases agree with PuLP/CBC.")

problem                 HiPO obj      PuLP obj     rel gap   HiPO viol   HiPO ms   PuLP ms
------------------------------------------------------------------------------------------
mixed_seed_0            -10.1248      -10.1248   9.889e-10   4.852e-10     3.028     2.915
mixed_seed_1            -11.6599      -11.6599   6.967e-09   4.294e-09     4.477     2.385
mixed_seed_2            -15.1744      -15.1744   2.176e-09   2.513e-09     4.659     2.314

PASS: all 3 mixed-sense cases agree with PuLP/CBC.


[IPM] Solver: HiPOLDLT (multifrontal BK+iterative refinement)
[IPM] Solver: HiPOLDLT (multifrontal BK+iterative refinement)
[IPM] Solver: HiPOLDLT (multifrontal BK+iterative refinement)
